# LangGraph Supervisor + Workers Demo (Gemini Edition)

This notebook demonstrates a multi-agent system using Google's Gemini model with LangGraph.

## Architecture

A central **Supervisor** node routes between specialist **Worker** nodes:
- **Researcher**: Gathers facts and information
- **Writer**: Composes the final answer

The supervisor makes routing decisions dynamically based on the conversation state.

## Setup

Before running this notebook, ensure you have:
1. Google's Generative AI API key (from https://ai.google.dev)
2. Required packages installed: `pip install langgraph langchain-google-genai langchain-core`
3. Environment variable set: `export GOOGLE_API_KEY="your-key-here"`

In [ ]:
# Install required packages
import subprocess
import sys

packages = ["langgraph", "langchain-google-genai", "langchain-core"]
for package in packages:
    try:
        __import__(package.replace("-", "_"))
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

print("✓ All packages installed")

In [ ]:
# Import required libraries
import os
from typing import Literal, TypedDict, Annotated

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# Check for API key
api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
    print("⚠️  GOOGLE_API_KEY not set. Please set it before running the demo.")
    print("Get your key at: https://ai.google.dev")

In [ ]:
# Define the agent state
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    next: str  # which node the supervisor wants to run next

# Initialize Gemini LLM
model_name = os.environ.get("GEMINI_MODEL", "gemini-2.0-flash")
api_key = os.environ.get("GOOGLE_API_KEY")

if api_key:
    llm = ChatGoogleGenerativeAI(
        model=model_name,
        temperature=0,
        api_key=api_key,
    )
    print(f"✓ Gemini LLM initialized with model: {model_name}")
else:
    llm = None
    print("⚠️  Cannot initialize LLM without GOOGLE_API_KEY")

In [ ]:
# Define worker nodes
def researcher_node(state: AgentState) -> dict:
    """Gathers facts and information about the topic."""
    system_prompt = SystemMessage(content=(
        "You are a Research agent. Given the conversation so far, produce "
        "a concise set of relevant facts or considerations on the topic. "
        "Do not write a final answer - just gather and list what's relevant. "
        "Prefix your response with 'RESEARCH NOTES:'."
    ))
    response = llm.invoke([system_prompt] + state["messages"])
    return {"messages": [AIMessage(content=response.content, name="researcher")]}


def writer_node(state: AgentState) -> dict:
    """Writes the final answer using gathered research."""
    system_prompt = SystemMessage(content=(
        "You are a Writer agent. Using the research notes already present "
        "in the conversation, write a clear, final answer for the user. "
        "Prefix your response with 'FINAL ANSWER:'."
    ))
    response = llm.invoke([system_prompt] + state["messages"])
    return {"messages": [AIMessage(content=response.content, name="writer")]}

print("✓ Worker nodes defined: researcher, writer")

In [ ]:
# Define the supervisor node
WORKERS = ["researcher", "writer"]
OPTIONS = WORKERS + ["FINISH"]

def supervisor_node(state: AgentState) -> dict:
    """Routes to the next worker or finishes execution."""
    system_prompt = SystemMessage(content=(
        f"You are a Supervisor managing these workers: {WORKERS}.\n"
        "Given the conversation, decide who should act next.\n"
        "- If there are no RESEARCH NOTES yet, choose 'researcher'.\n"
        "- If there are RESEARCH NOTES but no FINAL ANSWER yet, choose 'writer'.\n"
        "- If there is already a FINAL ANSWER, choose 'FINISH'.\n"
        f"Respond with exactly one word from this list: {OPTIONS}. "
        "No punctuation, no explanation - just the word."
    ))
    response = llm.invoke([system_prompt] + state["messages"])
    choice = response.content.strip()

    # Safety net
    if choice not in OPTIONS:
        choice = "FINISH"

    return {"next": choice}

print("✓ Supervisor node defined")

In [ ]:
# Define routing logic
def route(state: AgentState) -> Literal["researcher", "writer", "__end__"]:
    """Routes based on supervisor decision."""
    if state["next"] == "FINISH":
        return "__end__"
    return state["next"]

print("✓ Routing logic defined")

In [ ]:
# Build the graph
def build_graph():
    graph = StateGraph(AgentState)

    graph.add_node("supervisor", supervisor_node)
    graph.add_node("researcher", researcher_node)
    graph.add_node("writer", writer_node)

    # Workers report back to supervisor
    graph.add_edge("researcher", "supervisor")
    graph.add_edge("writer", "supervisor")

    # Conditional routing from supervisor
    graph.add_conditional_edges(
        "supervisor",
        route,
        {"researcher": "researcher", "writer": "writer", "__end__": END},
    )

    graph.set_entry_point("supervisor")

    return graph.compile()

if llm:
    app = build_graph()
    print("✓ Graph compiled successfully")
else:
    print("⚠️  Graph compilation skipped (no LLM available)")

In [ ]:
# Run the demo
if llm and app:
    user_question = "What is machine learning?"
    
    print(f"\n📝 Running demo with question: {user_question}\n")
    
    result = app.invoke(
        {"messages": [HumanMessage(content=user_question)], "next": ""},
        config={"recursion_limit": 10},
    )
    
    print("=== FULL CONVERSATION TRACE ===\n")
    for msg in result["messages"]:
        speaker = getattr(msg, "name", None) or msg.type
        print(f"[{speaker}]")
        print(msg.content)
        print()
else:
    print("⚠️  Demo skipped: LLM not initialized. Set GOOGLE_API_KEY to run.")

## How to Run This Notebook

1. **Set up your Google API Key:**
   ```bash
   export GOOGLE_API_KEY="your-key-here"
   ```

2. **Run all cells** to execute the multi-agent workflow

3. **Modify the question** in the last cell to test different inputs:
   ```python
   user_question = "Your custom question here"
   ```

## Key Concepts

- **Supervisor Pattern**: Central routing node decides which worker acts next
- **Message Reducer**: `add_messages` accumulates conversation history
- **Conditional Edges**: Routes dynamically based on state
- **State Flow**: Each node receives full state and returns partial updates

## Comparing Gemini vs Groq

See `supervisor_demo_groq.py` for the Groq version. Main differences:
- Import: `langchain_google_genai` vs `langchain_groq`
- LLM class: `ChatGoogleGenerativeAI` vs `ChatGroq`
- API Key: `GOOGLE_API_KEY` vs `GROQ_API_KEY`
- Models available differ between providers